# End-to-end system identification: a user-defined model

This notebook takes a pharmacokinetic model from a plain Python function all the way to
confidence intervals on its parameters, using a **user-defined** model.

The point is not that the fit succeeds. It is that **a good fit tells you almost nothing about
whether your parameters are identifiable** — and that the toolbox will tell you the difference
if you ask it.

A MATLAB Live Script version of this example (`Examples/pk_user_defined.m`) runs the same
computation with the same section numbering, so the two can be read side by side. Throughout,
`# MATLAB:` comments give the equivalent toolbox call.

## The model

A one-compartment model with first-order absorption, the standard description of an orally
administered drug:

$$c(t)=\frac{D\,k_a}{V(k_a-k_e)}\left(e^{-k_e t}-e^{-k_a t}\right)$$

Three factors are to be identified, with the dose $D=100$ mg known:

- $k_a$ — absorption rate constant (1/h)
- $k_e$ — elimination rate constant (1/h)
- $V$ — apparent volume of distribution (L)

**This is the whole model.** Everything below calls this one function; nothing else evaluates
the pharmacokinetics.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gsua_csb import (UserFunctionModel, parameter_estimation,
                      identifiability_analysis, profile_likelihood)

DOSE = 100.0        # administered dose (mg), known


def pk_absorption(params, t):
    """Plasma concentration after a single oral dose.

    params : (3,) array of [ka, ke, V]
        ka  absorption rate constant (1/h)
        ke  elimination rate constant (1/h)
        V   apparent volume of distribution (L)
    t : (n,) array of sampling times, in hours

    Returns
    -------
    (1, n) array -- one observed output (the concentration) over the n times.
    The leading axis is the output index: gsua_csb models always return
    (n_outputs, n_times), even when there is only one output.

    MATLAB equivalent: Examples/pkAbsorptionModel.m, which returns an
    ODE-solver-shaped struct so gsua_eval can interpolate it.
    """
    ka, ke, V = params
    return ((DOSE * ka) / (V * (ka - ke)) * (np.exp(-ke * t) - np.exp(-ka * t)))[None, :]


# Sanity check: evaluate the model once at the values we will try to recover.
pk_absorption(np.array([1.2, 0.25, 15.0]), np.array([1.0, 4.0, 12.0])).round(3)

## 1. Preparing the environment

`UserFunctionModel` wraps the function above into the object every other `gsua_csb` routine
consumes — the counterpart of the summary table `T` that `gsua_dataprep` builds in MATLAB.

The $k_a$ bounds are deliberately kept above the $k_e$ bounds. At $k_a=k_e$ the closed form is
singular, and swapping the two leaves $c(t)$ unchanged — the classic *flip-flop* ambiguity.
Excluding it keeps this example about experimental design rather than an algebraic accident.

In [ ]:
# Factor bounds: one row per factor, [lower, upper] -- same layout as MATLAB's `ranges`.
ranges = np.array([[0.6,  3.0],     # ka  absorption rate (1/h)
                   [0.05, 0.5],     # ke  elimination rate (1/h)
                   [5.0,  40.0]])   # V   volume of distribution (L)

truth = np.array([1.2, 0.25, 15.0])                        # values to recover
xdata = np.array([0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8,
                  10, 12, 16, 20, 24.0])                   # sampling schedule (hours)

# MATLAB: [T,~] = gsua_dataprep('pkAbsorptionModel', ranges, 'domain',[0 24], ...)
model = UserFunctionModel(
    func=pk_absorption,          # <-- the model function defined above
    names=["ka", "ke", "V"],
    range=ranges,
    nominal=truth,
    domain=xdata,
    output_names=["concentration"],
)
print("factors:", model.names, "| free:", int((~model.fixed).sum()))

## 2. Synthetic data

Working from synthetic data means the truth is known, so the confidence intervals can be
*checked* rather than merely reported.

In [ ]:
rng = np.random.default_rng(0)                  # fix the noise draw so the page reproduces

clean = pk_absorption(truth, xdata)             # noise-free model output at the sample times
ydata = clean * (1 + 0.08 * rng.standard_normal(clean.shape))   # 8% proportional noise

tdense = np.linspace(0.05, 24, 300)             # dense grid, for drawing curves only

plt.figure(figsize=(6.4, 3.8))
plt.plot(tdense, pk_absorption(truth, tdense)[0], lw=1.6, label="true model")
plt.plot(xdata, ydata[0], "ko", ms=5, label="measurements")
plt.xlabel("time (h)"); plt.ylabel("concentration (mg/L)")
plt.title("Simulated single-dose concentration data")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 3. Estimating the parameters

`parameter_estimation` runs a multistart fit: `n=20` restarts the optimizer from twenty
different points in the factor space, which is how you find out whether the problem has one
solution or several.

In [ ]:
# margin=0.1 selects the correlation-penalized cost and records the margin on the
# result, so functions further down the pipeline can recover what was scored.
# MATLAB: [T3,res3] = gsua_pe(T, xdata, ydata, 'solver','lsqc', 'N',20, 'margin',0.1)
pe3 = parameter_estimation(model, xdata, ydata, n=20,
                           solver="least_squares", margin=0.1, seed=0)

best3 = pe3.x[np.argmin(pe3.cost)]              # pe3.x is (20, 3): one row per restart
pd.DataFrame({"true": truth, "estimated": best3.round(4)}, index=model.names)

In [ ]:
print(f"cost across the 20 restarts:  min {pe3.cost.min():.5g}   max {pe3.cost.max():.5g}")

plt.figure(figsize=(6.4, 3.8))
plt.plot(tdense, pk_absorption(best3, tdense)[0], lw=1.6, label="fitted model")
plt.plot(xdata, ydata[0], "ko", ms=5, label="measurements")
plt.xlabel("time (h)"); plt.ylabel("concentration (mg/L)")
plt.title(f"Fit with all three factors free (cost = {pe3.cost.min():.4g})")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

Every one of the twenty restarts converged to the same cost, and the fitted curve passes
cleanly through the data. On most projects this is where the analysis would stop.

## 4. Diagnosing identifiability

The correlation between the repeated estimates is the first warning sign. Values near $\pm 1$
mean the factors trade off against each other: many different combinations reproduce the same
curve.

In [ ]:
# MATLAB: array2table(corr(Est3'), ...)
ia3 = identifiability_analysis(model, pe3.x, cost=pe3.cost, cost_rtol=0.1, seed=0)
pd.DataFrame(ia3.correlation.round(4), index=model.names, columns=model.names)

In [ ]:
# profile_likelihood steps each factor away from the estimate while re-fitting all the
# others at every step, and reports where the fit stays statistically acceptable.
# MATLAB: gsua_likelihood(T3, xdata, ydata, 0.95, 0.05, 0.1, 0.01, 0.01, 15, 1, ...)
pl3 = profile_likelihood(model, xdata, ydata, alpha=0.95, margin=0.1)

pd.DataFrame({
    "CI_low":     pl3.range[:, 0].round(4),
    "CI_high":    pl3.range[:, 1].round(4),
    "width":      (pl3.range[:, 1] - pl3.range[:, 0]).round(4),
    "prior_low":  ranges[:, 0],
    "prior_high": ranges[:, 1],
}, index=model.names)

This is the result worth stopping on. The correlation between $k_a$ and $k_e$ is essentially
$-1$: the three factors cannot be separated from a single oral concentration curve, however
well that curve fits. That is a textbook pharmacokinetic result, not a failure of the
optimizer.

## 5. The remedy: fix what another experiment already knows

The standard resolution is to measure $V$ separately, in an intravenous study where it *is*
directly identifiable, and then estimate only the two rate constants. A factor is fixed by
giving it a degenerate range — lower bound equal to upper bound.

In [ ]:
ranges_fixed = ranges.copy()
ranges_fixed[2] = [15.0, 15.0]                  # V pinned at its known value

# MATLAB: same gsua_dataprep call with the V row given as [15 15]; there, fixed
# factors drop out of T entirely. Here they stay in the model and are reported by
# `model.fixed`, so indices remain stable -- a deliberate difference between the ports.
model_fixed = UserFunctionModel(
    func=pk_absorption, names=["ka", "ke", "V"], range=ranges_fixed,
    nominal=truth, domain=xdata, output_names=["concentration"],
)
free = np.where(~model_fixed.fixed)[0]          # -> [0, 1], i.e. ka and ke
print("free factors now:", [model.names[i] for i in free])

In [ ]:
pe2 = parameter_estimation(model_fixed, xdata, ydata, n=20,
                           solver="least_squares", margin=0.1, seed=0)
best2 = pe2.x[np.argmin(pe2.cost)]
ia2 = identifiability_analysis(model_fixed, pe2.x, cost=pe2.cost, cost_rtol=0.1, seed=0)
pl2 = profile_likelihood(model_fixed, xdata, ydata, alpha=0.95, margin=0.1,
                         params=list(free))    # profile only the free factors

pd.DataFrame({
    "true":      truth[free],
    "estimated": best2[free].round(4),
    "CI_low":    pl2.range[free, 0].round(4),
    "CI_high":   pl2.range[free, 1].round(4),
    "width":     (pl2.range[free, 1] - pl2.range[free, 0]).round(4),
}, index=[model.names[i] for i in free])

Both remaining factors are now recovered close to the truth, and both intervals sit well
inside their bounds instead of running to them.

## 6. The two runs side by side

Comparing the two fits on the quantities people usually conflate:

In [ ]:
pd.DataFrame({
    "all three free": [pe3.cost.min(), ia3.correlation[0, 1],
                       pl3.range[0, 1] - pl3.range[0, 0],
                       pl3.range[1, 1] - pl3.range[1, 0]],
    "V fixed":        [pe2.cost.min(), ia2.correlation[0, 1],
                       pl2.range[0, 1] - pl2.range[0, 0],
                       pl2.range[1, 1] - pl2.range[1, 0]],
}, index=["best cost", "corr(ka, ke)", "CI width ka", "CI width ke"]).round(4)

In [ ]:
x = np.arange(2); w = 0.35
plt.figure(figsize=(5.6, 3.6))
plt.bar(x - w/2, [pl3.range[i, 1] - pl3.range[i, 0] for i in range(2)], w, label="all three free")
plt.bar(x + w/2, [pl2.range[i, 1] - pl2.range[i, 0] for i in range(2)], w, label="V fixed")
plt.xticks(x, ["ka", "ke"]); plt.ylabel("95% confidence interval width")
plt.title("Fixing one factor sharpens the other two")
plt.legend(); plt.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.show()

## 7. What this example shows

Fixing $V$ made the fit slightly **worse** and the science considerably **better**: the
correlation between $k_a$ and $k_e$ collapses, both intervals tighten, and the estimates move
onto the truth.

Cost measures how well a curve passes through points. It does not measure whether the factors
that produced that curve could have been recovered. Only the identifiability analysis answers
that, which is why it belongs *inside* the workflow rather than after it.

> **Comparing with the MATLAB version.** The two profile-likelihood implementations use
> different thresholds and stepping strategies, so interval *widths* are not directly
> comparable between the languages — MATLAB reports $k_a$'s all-free interval as spanning its
> entire prior, this one reports a finite but wide interval. What transfers is the ordering and
> the conclusion: strongly correlated factors, intervals that tighten once $V$ is fixed.

The companion symbolic-ODE example reaches the same conclusion from the opposite direction:
there, the dataset that fits *better* is the one whose parameters are *less* identifiable.

Where a multistart run does spread across the factor space instead of converging to a single
point, `identifiability_analysis` adds correlation structure and detection of multiple global
minima, `noise_floor` calibrates which fits to accept against the observation noise, and
`design_matrix(..., method="joint")` propagates the accepted set without destroying its
correlation structure.